In [ ]:
import torch
import random

def encode(s: list, vocab: list) -> torch.tensor:
    """
    Кодирует список токенов в тензор целых чисел с использованием словаря.

    Каждый токен заменяется на его индекс в словаре. Если токена нет в словаре,
    используется специальный токен <UNK>. Если токен <UNK> тоже отсутствует в словаре,
    используется случайный индекс.

    Параметры:

    s : list
        Список токенов для кодирования
    vocab : list
        Словарь (список уникальных токенов)

    Возвращает:

    torch.tensor
        Тензор типа long с индексами токенов
    """
    # Генерируем случайный индекс на случай, если <UNK> нет в словаре
    rand_token = random.randint(0, len(vocab))  # Случайное число от 0 до len(vocab)

    # Определяем специальный токен для неизвестных слов
    unknown_token = "<UNK>"

    # Создаём словарь "токен -> индекс" из словаря vocab
    # Пример: vocab = ["кот", "собака"] -> {"кот": 0, "собака": 1}
    map = {token: idx for idx, token in enumerate(vocab)}

    # Кодируем каждый токен:
    # 1. Если токен есть в словаре -> берём его индекс
    # 2. Если нет -> пробуем взять индекс <UNK>
    # 3. Если <UNK> тоже нет -> используем случайный индекс
    enc = [map.get(token, map.get(unknown_token, rand_token)) for token in s]

    # Преобразуем список индексов в тензор PyTorch
    enc = torch.tensor(enc, dtype=torch.long)

    return enc

In [ ]:
from typing import List, Set, Union
from nltk.tokenize import RegexpTokenizer
from collections import Counter

def custom_tokenizer(txt: str, spec_tokens: List[str], pattern: str = r"|\d|\w+|[^\s]") -> List[str]:
    """
    Токенизирует текст с учётом специальных токенов.

    Токенизация выполняется с помощью регулярных выражений (NLTK RegexpTokenizer).
    Специальные токены сохраняются как целые единицы и не разбиваются.

    Параметры:

    txt : str
        Исходный текст для токенизации
    spec_tokens : List[str]
        Список специальных токенов (например, <END>, <UNK>),
        которые должны оставаться неделимыми
    pattern : str, optional
        Шаблон для токенизации (по умолчанию: слова, числа и знаки пунктуации).
        Для посимвольной токенизации используйте '|.'

    Возвращает:

    List[str]
        Список токенов

    Примечания:

    1. Специальные токены добавляются в начало шаблона, поэтому имеют приоритет
    2. Пробелы игнорируются и служат разделителями
    """
    pattern = "|".join(spec_tokens) + pattern

    # создадим объект-токенизатор на основе заданного регулярного выражения
    tokenizer = RegexpTokenizer(pattern)
    # выполним токенизацию
    tokens = tokenizer.tokenize(txt)
    return tokens


def get_infrequent_tokens(tokens: Union[List[str], str], min_count: int) -> List[str]:
    """
    Возвращает токены, которые встречаются реже заданного порога.

    Функция подсчитывает частоту встречаемости токенов и возвращает те,
    количество которых меньше или равно указанному порогу.

    Параметры:

    tokens : Union[List[str], str]
        Если передана строка, подсчёт частоты выполняется посимвольно.
        Если передан список, подсчёт выполняется по токенам.
    min_count : int
        Пороговое значение встречаемости. Токены с частотой <= этому значению
        считаются редкими.

    Возвращает:

    List[str]
        Список редких токенов
        Возвращаются уникальные токены (без дубликатов)
    """
    counts = Counter(tokens)  # Подсчитываем частоту каждого токена
    infreq_tokens = set([k for k,v in counts.items() if v <= min_count])  # Фильтруем редкие
    return infreq_tokens  # Возвращаем список редких токенов

def mask_tokens(tokens: List[str], mask: Set[str]) -> List[str]:
    """
    Проходим по всем токенам.
    Если токен в множестве редких токенов, то он заменяется на <UNK>
    """
    return ["<UNK>" if i in mask else i for i in tokens]



In [ ]:
import copy
import torch
import torch.nn as nn
import math

# Устройство, преносим на куду (граф процессор), у меня его нет,
# так что код будет выполняться на моём бедненьком Intel Core i5-10210U CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scaled Dot-Product Attention
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, _ = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

# Position-wise Feed Forward
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0, max_len, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

# Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.src_attn(x, memory, memory, src_mask)))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# Энкодер
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Декодер
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

# Полная модель
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, N=2, d_ff=512, h=4, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Sequential(
            nn.Embedding(src_vocab, d_model),
            PositionalEncoding(d_model, dropout)
        )
        self.tgt_embed = nn.Sequential(
            nn.Embedding(tgt_vocab, d_model),
            PositionalEncoding(d_model, dropout)
        )

        self.encoder = Encoder(
            EncoderLayer(d_model, MultiHeadAttention(h, d_model, dropout),
                         PositionwiseFeedForward(d_model, d_ff, dropout), dropout),
            N
        )

        self.decoder = Decoder(
            DecoderLayer(d_model, MultiHeadAttention(h, d_model, dropout),
                         MultiHeadAttention(h, d_model, dropout),
                         PositionwiseFeedForward(d_model, d_ff, dropout), dropout),
            N
        )

        self.out = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, src_mask, tgt_mask):
        src = src.to(device)
        tgt = tgt.to(device)
        if src_mask is not None:
            src_mask = src_mask.to(device)
        if tgt_mask is not None:
            tgt_mask = tgt_mask.to(device)

        memory = self.encoder(self.src_embed(src), src_mask)
        output = self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
        return self.out(output)  # возвращаем логиты (без softmax — CrossEntropyLoss применяет его сам)
    import torch
import torch.nn as nn
import math

# Устройство, преносим на куду (граф процессор), у меня его нет,
# так что код будет выполняться на моём бедненьком Intel Core i5-10210U CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scaled Dot-Product Attention
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

# Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(4)])
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, _ = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)

# Position-wise Feed Forward
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0, max_len, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

# Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.src_attn(x, memory, memory, src_mask)))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# Энкодер
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Декодер
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = nn.LayerNorm(layer.norm1.normalized_shape)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

# Полная модель
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, N=2, d_ff=512, h=4, dropout=0.1):
        super().__init__()
        self.src_embed = nn.Sequential(
            nn.Embedding(src_vocab, d_model),
            PositionalEncoding(d_model, dropout)
        )
        self.tgt_embed = nn.Sequential(
            nn.Embedding(tgt_vocab, d_model),
            PositionalEncoding(d_model, dropout)
        )

        self.encoder = Encoder(
            EncoderLayer(d_model, MultiHeadAttention(h, d_model, dropout),
                         PositionwiseFeedForward(d_model, d_ff, dropout), dropout),
            N
        )

        self.decoder = Decoder(
            DecoderLayer(d_model, MultiHeadAttention(h, d_model, dropout),
                         MultiHeadAttention(h, d_model, dropout),
                         PositionwiseFeedForward(d_model, d_ff, dropout), dropout),
            N
        )

        self.out = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, src_mask, tgt_mask):
        src = src.to(device)
        tgt = tgt.to(device)
        if src_mask is not None:
            src_mask = src_mask.to(device)
        if tgt_mask is not None:
            tgt_mask = tgt_mask.to(device)

        memory = self.encoder(self.src_embed(src), src_mask)
        output = self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)
        return self.out(output)  # возвращаем логиты (без softmax — CrossEntropyLoss применяет его сам)


In [ ]:
import torch
import os

# Загружаем наш текст
text = [str(i).strip('\n') for i in open("book_clean.txt")]

# Соединяем в одну строку с разделителем <END>
corpus = ' <END> '.join(text)

# Список спецтокенов
# <END> - разделитель
# <UNK> - редкие слова
spec_tokens = ["<SOS>", "<END>", "<UNK>"]  # добавлен отдельный токен <SOS>

# Токенизация

tokens = custom_tokenizer(corpus, spec_tokens) # разобьём (токенизируем) текст
infreq_tokens = get_infrequent_tokens(tokens, min_count=2) # находим токены, которые встречаются меньше, чем 2 раза
tokens = mask_tokens(tokens, infreq_tokens) # заменим их на <UNK>

print("Всего токенов:", len(tokens))
print("Всего уникальных токенов:", len(set(tokens)))

# Словарь

# Создаём список уникальных токенов из корпуса
# set(tokens) удаляет дубликаты, list() превращает обратно в список
vocab_list = sorted(set(tokens))  # sorted() для стабильного порядка словаря между запусками

# Создаём словарь "слово -> индекс"
# Каждому уникальному токену присваиваем числовой индекс
word2idx = {w: i for i, w in enumerate(vocab_list)}

# Создаём обратный словарь "индекс -> слово"
# Позволяет по числовому индексу получить исходный токен
idx2word = {i: w for w, i in word2idx.items()}

# кодируем список токенов в тензор целых чисел
enc = encode(tokens, vocab_list)
print(enc)

# Разделение на батчи

# Длина последовательности (количество токенов в одном примере)
seq_len = 16

# Разбиваем закодированный текст на куски по seq_len токенов
# enc - это список или тензор с закодированными токенами
# Создаем список из последовательностей длиной 16 токенов
chunks = [enc[i:i+seq_len] for i in range(0, len(enc), seq_len)]

# Оставляем только полные последовательности (длиной ровно 16)
# Это важно, чтобы все примеры в батче имели одинаковую длину
chunks = [c for c in chunks if len(c) == seq_len]

# Преобразуем список последовательностей в тензор PyTorch
# Получаем тензор размерности (batch_size, seq_len)
# где batch_size - количество 16-токенных последовательностей
src = torch.stack(chunks)  # (batch, seq_len)

# Получаем индекс токена начала последовательности (START OF SEQUENCE)
# Внимание: у вас странное название переменной - SOS обычно означает Start,
# но вы используете <END> токен. Возможно, это опечатка или особенность вашей модели?
SOS_idx = word2idx["<SOS>"]  # токен начала последовательности

# Создаем целевые последовательности (tgt) для обучения
# Для каждой последовательности в src:
# 1. Создаем тензор из одного токена SOS в начале
# 2. Добавляем все токены из src, кроме последнего (src[:, :-1])
# Это стандартный подход для обучения language models:
# модель учится предсказывать следующий токен по предыдущим
tgt = torch.cat([torch.full((src.size(0), 1), SOS_idx), src[:, :-1]], dim=1)


# Маски


# Маска источника (src_mask)

# Используется в Encoder для игнорирования padding-токенов
# Но в вашем случае, так как у вас нет padding (все последовательности одинаковой длины),
# эта маска просто разрешает всем позициям
src_mask = torch.ones(src.size(0), 1, src.size(1), dtype=torch.bool)

# Размеры: (batch_size, 1, seq_len)
# Например: (1331, 1, 16) - для всех 1331 батчей и всех 16 позиций
# Форма (batch, 1, seq_len) позволяет broadcast при умножении с вниманием
# True/1 означает "разрешить", False/0 - "замаскировать" (игнорировать)

# Если бы у вас был padding, маска выглядела бы так:
# [[[True, True, True, ..., False, False, False]]] - где False для padding токенов


# Маска цели (tgt_mask)

# Используется в Decoder для предотвращения "подглядывания в будущее" (causal masking)
# Каждый токен может видеть только предыдущие токены и себя
tgt_mask = torch.tril(torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool)).unsqueeze(0)

# Разберем по частям:
# 1. torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool)
#    Создает квадратную матрицу размером (seq_len, seq_len) из True
#    Например, для seq_len=4:
#    [[True, True, True, True],
#     [True, True, True, True],
#     [True, True, True, True],
#     [True, True, True, True]]

# 2. torch.tril(...) - берет нижний треугольник (lower triangular)
#    Оставляет True только ниже и на главной диагонали
#    После tril:
#    [[True, False, False, False],
#     [True, True,  False, False],
#     [True, True,  True,  False],
#     [True, True,  True,  True]]
#
#    Это означает:
#    - Токен 0 видит только себя
#    - Токен 1 видит токены 0 и 1
#    - Токен 2 видит токены 0, 1 и 2
#    - Токен 3 видит все токены

# 3. .unsqueeze(0) - добавляет размерность batch
#    Размер становится: (1, seq_len, seq_len)
#    Это позволяет использовать одну маску для всех примеров в батче через broadcast

# Такая маска гарантирует, что при предсказании токена i
# модель использует только токены 0..i-1 (не "подглядывая" в будущее)

# Пример

# Предположим, seq_len = 4:
# tgt: [<SOS>, "привет", "как", "дела"]
#
# При вычислении внимания для каждого токена:
# - "<SOS>" → видит: [<SOS>]                          (предсказывает "привет")
# - "привет" → видит: [<SOS>, "привет"]               (предсказывает "как")
# - "как" → видит: [<SOS>, "привет", "как"]           (предсказывает "дела")
# - "дела" → видит: [<SOS>, "привет", "как", "дела"] (предсказывает <EOS>)



# Инициализация модели

# Определяем устройство для вычислений (GPU или CPU)
# Приоритет отдается CUDA (GPU Nvidia), если доступно
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

# Определяем размер словаря
# В языковой модели source и target словари обычно одинаковые
src_vocab = len(vocab_list)  # Размер словаря для encoder
tgt_vocab = len(vocab_list)  # Размер словаря для decoder

# Создаем экземпляр модели Transformer
model = Transformer(src_vocab, tgt_vocab).to(device)

# Функция потерь для многоклассовой классификации
# CrossEntropyLoss объединяет LogSoftmax и NLLLoss
# Используется потому что на выходе модели: (batch_size, seq_len, vocab_size)
# А нам нужно предсказать следующий токен для каждой позиции
criterion = torch.nn.CrossEntropyLoss()

# Оптимизатор Adam - популярный адаптивный метод градиентного спуска
# lr=1e-4 (0.0001) - стандартная начальная скорость обучения для трансформеров
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),  # стандартные для трансформеров
    eps=1e-9  # для численной стабильности
)

# Перенос данных на выбранное устройство (GPU/CPU)
# Важно: данные и модель должны быть на одном устройстве
src, tgt = src.to(device), tgt.to(device)

# Перенос масок на устройство
src_mask, tgt_mask = src_mask.to(device), tgt_mask.to(device)


# Сохранение

# Путь к файлу сохранения
checkpoint_path = "transformer_checkpoint.pth"

# Загрузка, если есть сохранение
start_epoch = 1
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Загружено сохранение: эпоха {start_epoch - 1}")



# Обучение модели

# Количество эпох - сколько раз модель увидит весь набор данных
epochs = 2500

# start_epoch позволяет продолжить обучение с сохраненной точки
for epoch in range(start_epoch, epochs + 1):

    model.train()

    # Обнуляем градиенты перед каждым шагом обучения
    # Градиенты накапливаются, если их не обнулять
    optimizer.zero_grad()

    # Прямой проход: подаем данные в модель
    # Модель возвращает предсказания для каждого токена в последовательности
    # Размер выхода: (batch_size, длина_последовательности, размер_словаря)
    out = model(src, tgt, src_mask, tgt_mask)  # (batch, seq_len, vocab)

    # Вычисляем функцию потерь
    # Преобразуем выход модели и целевые токены для CrossEntropyLoss
    # view(-1, tgt_vocab) превращает 3D тензор в 2D: (batch*seq_len, vocab_size)
    # tgt.view(-1) превращает целевые токены в 1D: (batch*seq_len)
    loss = criterion(out.view(-1, tgt_vocab), tgt.view(-1))

    # Обратный проход: вычисляем градиенты
    # PyTorch автоматически вычисляет производные по всей вычислительной графе
    loss.backward()

    # Шаг оптимизации: обновляем веса модели
    # Оптимизатор использует вычисленные градиенты для изменения параметров
    optimizer.step()

    # Выводим информацию о текущей эпохе
    # .item() извлекает числовое значение из тензора loss
    print(f"Эпоха {epoch}/{epochs} | Loss: {loss.item():.4f}")

    # Сохранение контрольной точки (checkpoint)
    # Сохраняем состояние обучения, чтобы можно было продолжить позже
    torch.save({
        "epoch": epoch,  # Текущая эпоха
        "model_state_dict": model.state_dict(),  # Веса модели
        "optimizer_state_dict": optimizer.state_dict(),  # Состояние оптимизатора
        "loss": loss.item()  # Значение функции потерь
    }, checkpoint_path)
    print(f"Сохранено в {checkpoint_path}")




Всего токенов: 21301
Всего уникальных токенов: 855
tensor([374, 374, 100,  ..., 838, 653, 223])
Используется устройство: cuda
Загружено сохранение: эпоха 2431
Эпоха 2432/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2433/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2434/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2435/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2436/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2437/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2438/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2439/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2440/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2441/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2442/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Эпоха 2443/2500 | Loss: 5.7531
Сохранено в transformer_checkpoint.pth
Э

In [ ]:
# Тест генерации
def generate_from_enc(model, enc_tensor, idx2word, SOS_idx, max_len=10):
    model.eval()
    src = enc_tensor.unsqueeze(0).to(device)  # (1, seq_len)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool, device=device)

    generated = torch.full((1, 1), SOS_idx, dtype=torch.long, device=device)
    for _ in range(max_len):
        tgt_mask = torch.tril(torch.ones(generated.size(1), generated.size(1), dtype=torch.bool, device=device)).unsqueeze(0)
        probs = model(src, generated, src_mask, tgt_mask)

        probs = torch.softmax(probs[:, -1, :] / 0.8, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_token], dim=1)

    tokens_out = [idx2word[int(i)] for i in generated[0, 1:]]  # без SOS
    return tokens_out


test_seq = chunks[random.randint(0, len(chunks)-1)]
gen_tokens = generate_from_enc(model, test_seq, idx2word, SOS_idx)
print("Сгенерированный текст:", " ".join(gen_tokens))

Сгенерированный текст: Еще тот свой мой друг которых хотел заговорить Марк землю


In [ ]:
# Функция общения с моделью
def chat_with_model(model, word2idx, idx2word, SOS_idx, seq_len=16, max_gen_len=10):
    print("Начинаем чат с моделью! (введите 'exit' для выхода)")
    model.eval()

    history_tokens = []  # для накопления контекста

    while True:
        user_input = input("Вы: ")
        if user_input.lower() == "exit":
            break

        # Токенизация и замена редких слов
        user_tokens = custom_tokenizer(user_input, spec_tokens)
        user_tokens = mask_tokens(user_tokens, infreq_tokens)

        history_tokens.extend(user_tokens)
        # Оставляем только последние seq_len токенов
        context_tokens = history_tokens[-seq_len:]

        # Кодировка
        context_enc = encode(context_tokens, vocab_list)
        context_enc = torch.tensor(context_enc, dtype=torch.long).to(device) if not isinstance(context_enc, torch.Tensor) else context_enc.detach().clone().long().to(device)


        # Генерация ответа
        gen_tokens = generate_from_enc(model, context_enc, idx2word, SOS_idx, max_len=max_gen_len)

        print("Модель:", " ".join(gen_tokens))

        # Добавляем ответ модели в контекст для поддержания диалога
        history_tokens.extend(gen_tokens)


# Запуск чата
chat_with_model(model, word2idx, idx2word, SOS_idx)

Начинаем чат с моделью! (введите 'exit' для выхода)
Вы: Привет!
Модель: французски выход занятие еще телек хотел небоскребы чего легче »
Вы: какие небоскрёбы?
Модель: пять будет музыку хочешь не себя Когда тело людей ладони
Вы: хочу
Модель: Таймс тут одним Пожалуйста шестой тех бабушка 1 наш миром
Вы: ты читаешь таймс
Модель: носить чем наручные том Как ma нож рассказать отсутствие песня
Вы: любишь музыку
Модель: вдруг тебя почти Мое страданий Бабушка часы сложные Пап своим
Вы: exit
